In [ ]:
#Imports:

import pandas as pd
import numpy as np
import urllib.request
import zipfile
import os

In [ ]:
#1. Configuração do diretório:

data_path = "../data/nasa"
os.makedirs(data_path, exist_ok=True)

In [ ]:
#2. Download do Dataset:

print("Baixando dataset...")

url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00278/CMAPSSData.zip"
zip_path = os.path.join(data_path, "CMAPSSData.zip")

Baixando dataset da NASA...


In [ ]:
#3.Definindo o nome das colunas:

col_names = ['engine_id', 'time_cycle', 'op_setting_1', 'op_setting_2', 'op_setting_3']
sensor_cols = [f'sensor_{i}' for i in range(1, 22)]
col_names.extend(sensor_cols)

In [ ]:
#4. ETL:

train_file = os.path.join(data_path, "train_FD001.txt")

try:
    df_train = pd.read_csv(train_file, sep='\s+', header=None, names=col_names)
    print(f"Dados carregados: {df_train.shape[0]} linhas e {df_train.shape[1]} colunas.")
    
    #Descobre qual foi o último ciclo (tempo de falha) para cada motor
    max_cycles = df_train.groupby('engine_id')['time_cycle'].max().reset_index()
    max_cycles.columns = ['engine_id', 'max_cycle']
    
    #Junta com o dataset original
    df_train = df_train.merge(max_cycles, on=['engine_id'], how='left')
    
    #Calcula o RUL: (Ciclo Máximo do Motor) - (Ciclo Atual)
    df_train['RUL'] = df_train['max_cycle'] - df_train['time_cycle']
    
    #Remove a coluna auxiliar
    df_train.drop('max_cycle', axis=1, inplace=True)
    
    print("\nVisualização das primeiras linhas com a nova coluna RUL:")
    display(df_train[['engine_id', 'time_cycle', 'sensor_2', 'sensor_3', 'RUL']].head())
    
    #Salva o dado processado
    df_train.to_csv(os.path.join(data_path, "nasa_cmapss_fd001_processed.csv"), index=False)
    print("\nDataset processado salvo com sucesso em data/nasa")

except FileNotFoundError:
    print("Baixar o arquivo CMAPSSData.zip, extrair e colocar o 'train_FD001.txt' na pasta 'data/nasa'")

Dados carregados: 20631 linhas e 26 colunas.

Visualização das primeiras linhas com a nova coluna RUL:


,engine_id,time_cycle,sensor_2,sensor_3,RUL
0,1,1,641.82,1589.70,191
1,1,2,642.15,1591.82,190
2,1,3,642.35,1587.99,189
3,1,4,642.35,1582.79,188
4,1,5,642.37,1582.85,187



Dataset processado salvo com sucesso em data/nasa!
